# 📊 DataAn Enhanced — Google Colab

Полный автоматический **статистический анализ** данных с генерацией интерактивного **HTML-отчёта** (Plotly).

## Как пользоваться
1. **Меню → Среда выполнения → Сменить среду выполнения** → выберите *GPU* или *CPU* (GPU не обязателен).
2. Запустите все ячейки: **Среда выполнения → Выполнить все**.
3. В ячейке **📤 Загрузка данных** нажмите **«Загрузить файл»** и выберите файл `.xlsx` / `.xls` / `.csv`.
4. Запустите ячейку **🚀 Анализ** дважды: первый раз — чтобы выбрать параметры (виджеты), второй — чтобы выполнить анализ.
5. Отчёт сохраняется как `*_report.html`. Кнопка **«📂 Открыть HTML»** скачает его в браузер.

## Требования к данным
- Файл должен содержать **не менее 1 категориального** и **не менее 1 количественного** столбца.
- При нескольких листах анализируется **только первый**.
- Если файл уже содержит столбец `param_group` — он будет использован как группирующий.



In [ ]:
# -*- coding: utf-8 -*-
#@title ⚙️ Установка и подготовка окружения
import os, sys, io, gc, shutil, subprocess
import pandas as pd

# --- Удаление myenv, если остался от предыдущих запусков ---
if os.path.isdir('myenv'):
    shutil.rmtree('myenv', ignore_errors=True)
    print('Удалена директория myenv')

# --- Установка зависимостей ---
!pip install -q plotly xgboost statsmodels scikit-learn ipywidgets openpyxl 2>/dev/null

# --- Восстановление актуальной версии analyzer_enhanced.py (встроена в ноутбук) ---
import base64
_ANALYZER_B64 = (
    "..."
)

with open('analyzer_enhanced.py', 'w', encoding='utf-8') as _f:
    _f.write(base64.b64decode("".join(_ANALYZER_B64)).decode('utf-8'))

import warnings
warnings.filterwarnings('ignore')
import importlib, analyzer_enhanced
importlib.reload(analyzer_enhanced)
from analyzer_enhanced import DataAnalyzer
print('Окружение готово ✅')

#@title 📤 Загрузка данных
def create_interactive_file_uploader():
    import io as _io
    import ipywidgets as widgets
    from IPython.display import display, HTML, clear_output
    _state = {'df': None, 'filename': None}
    upload_widget = widgets.FileUpload(
        accept='.xlsx,.xls,.csv', multiple=False, description='Загрузить файл'
    )
    out = widgets.Output()

    def on_upload_change(change):
        with out:
            clear_output(wait=True)
            if not upload_widget.value:
                _state['df'] = None
                _state['filename'] = None
                return
            try:
                upload_info = upload_widget.value
                if isinstance(upload_info, dict):
                    file_info = next(iter(upload_info.values()))
                elif isinstance(upload_info, (list, tuple)) and len(upload_info) > 0:
                    file_info = upload_info[0]
                else:
                    file_info = upload_info
                fname = file_info.get('name', '')
                if not fname:
                    meta = file_info.get('metadata', {})
                    fname = meta.get('name', 'uploaded_file')
                content = file_info.get('content', None)
                if content is None:
                    data = file_info.get('data', None)
                    if data is not None:
                        content = _io.BytesIO(data) if isinstance(data, (bytes, bytearray)) else _io.BytesIO(bytes(data))
                if content is None:
                    print('Не удалось прочитать содержимое файла.')
                    return
                if hasattr(content, 'tobytes'):
                    content = _io.BytesIO(content.tobytes())
                elif isinstance(content, (bytes, bytearray)):
                    content = _io.BytesIO(content)
                if fname.endswith('.csv'):
                    _state['df'] = pd.read_csv(content)
                else:
                    _state['df'] = pd.read_excel(content)
                _state['filename'] = fname
                print(f'Загружен: {fname} ({_state["df"].shape[0]} строк, {_state["df"].shape[1]} столбцов)')
                display(_state['df'].head(10))
            except Exception as e:
                print(f'Ошибка загрузки: {e}')
                _state['df'] = None
                _state['filename'] = None

    upload_widget.observe(on_upload_change, names='value')
    display(widgets.VBox([
        widgets.HTML('<h3>Загрузка данных</h3>'),
        upload_widget, out
    ]))

    def get_uploaded_data():
        return _state['df']

    def get_file_name():
        return _state['filename'] or 'Uploaded_Data'

    return get_uploaded_data, get_file_name

get_uploaded_data, get_file_name = create_interactive_file_uploader()



In [ ]:
# -*- coding: utf-8 -*-
#@title 🚀 Анализ и HTML-отчёт
"""
ЯЧЕЙКА 2: Полный цикл анализа и генерация HTML-отчёта.
"""
try:
    uploaded_df = get_uploaded_data()
    uploaded_file_name = get_file_name()
    if uploaded_df is None:
        raise ValueError("Файл не загружен. Используйте виджет в Ячейке «Загрузка данных».")

    need_setup = 'analyzer' not in globals() or not getattr(analyzer, 'params', None)
    if need_setup:
        if 'analyzer' not in globals():
            analyzer = DataAnalyzer(uploaded_df, file_name=uploaded_file_name)
        analyzer.create_parameter_selector()
        print("⚠️ Выберите параметры в виджетах выше и нажмите 'Применить', затем запустите эту ячейку ЕЩЁ РАЗ.")
    else:
        import ipywidgets as widgets
        from IPython.display import display, clear_output, HTML, FileLink

        print('='*60)
        print('🚀 ЗАПУСК ПОЛНОГО АНАЛИЗА ДАННЫХ')
        print('='*60)

        # --- ПРЕДОБРАБОТКА ---
        print('\n📌 Предварительная обработка...')
        df_clean = analyzer.preprocess(remove_outliers=True, z_threshold=3.0, balance_groups=True)
        analyzer._current_df = df_clean
        print(f'    Очищенные данные: {df_clean.shape[0]} строк, {df_clean.shape[1]} столбцов')

        # --- 1. ВИЗУАЛИЗАЦИЯ ---
        print('\n📊 1. Визуализации:')
        print('    1.1 Скрипичная диаграмма + диаграмма роя...')
        analyzer.plot_violin()
        print('    1.2 Ящик с усами (boxplot)...')
        analyzer.plot_boxplot_with_significance()
        print('    1.3 Гистограммы...')
        analyzer.plot_histograms()
        print('    1.4 Круговая диаграмма...')
        analyzer.plot_pie_chart()
        print('    1.5 Точечная диаграмма с линией регрессии...')
        analyzer.plot_scatter_with_regression()
        print('    1.6 Матрица корреляций...')
        analyzer.plot_correlation_matrix()

        # --- 2. СТАТИСТИЧЕСКИЙ АНАЛИЗ ---
        print('\n📊 2. ANOVA / критерий Крускала — Уоллиса:')
        print(analyzer.perform_anova_analysis())

        print('\n📊 3. Постхок-тест Тьюки:')
        print(analyzer.perform_posthoc_tukey())

        print('\n📊 4. Двухфакторный дисперсионный анализ (Two-way ANOVA):')
        print(analyzer.perform_two_way_anova())
        print('   График взаимодействия факторов:')
        analyzer.plot_interaction_effect()

        print('\n📊 5. Анализ категориальных данных:')
        print(analyzer.perform_categorical_analysis())

        print('\n📊 6. Многомерный дисперсионный анализ (MANOVA):')
        print(analyzer.perform_manova())

        print('\n📊 7. Постхок-анализ для MANOVA:')
        print(analyzer.perform_posthoc_manova())

        print('\n📊 8. Сравнение между выборками:')
        print(analyzer.perform_between_sample_comparison())

        # --- 3. РЕГРЕССИОННЫЙ АНАЛИЗ ---
        print('\n📊 9. Линейная регрессия:')
        print(analyzer.perform_linear_regression())
        print('   Диагностика модели:')
        analyzer.plot_regression_diagnostics()

        # --- 4. ОТБОР ПРИЗНАКОВ ---
        print('\n📊 10. Отбор признаков:')
        print('   Важность признаков по случайному лесу (RF):')
        print(analyzer.feature_selection_rf())
        print('   Рекурсивное исключение признаков (RFE):')
        print(analyzer.rfe_selection())
        print('   Метод главных компонент (PCA):')
        print(analyzer.pca_analysis())

        # --- 5. КЛАСТЕРНЫЙ АНАЛИЗ ---
        print('\n📊 11. Кластерный анализ:')
        print('   Метод локтя (подбор оптимального числа кластеров):')
        print(analyzer.determine_optimal_clusters(max_k=10))
        print('   Кластеризация методом K-средних (K-means):')
        print(analyzer.perform_kmeans())
        print('   ANOVA для кластеров:')
        print(analyzer.anova_for_clusters())
        cluster_xlsx = analyzer.save_clusters_to_xlsx()
        if cluster_xlsx:
            print(f'   Файл XLSX сохранён: {cluster_xlsx}')

        # --- 12. МАШИННОЕ ОБУЧЕНИЕ ---
        print('\n🤖 12. Машинное обучение:')
        analyzer.ml_benchmark(df_clean)

        print("\n" + "="*60)

        # --- НАСТРОЙКА HTML-ОТЧЁТА ---
        print("\n📄 Настройка HTML-отчёта...")
        analyzer.create_comment_widgets()

        sec_keys = [
            'plots', 'anova', 'manova',
            'linear_regression', 'feature_selection', 'pca',
            'cluster', 'ml',
        ]
        sec_labels = [
            'Визуализация (Violin, Boxplot, Гистограммы и др.)',
            'ANOVA / Kruskal-Wallis / Категориальные',
            'MANOVA / Post-hoc MANOVA / Межвыборочные',
            'Линейная регрессия',
            'Отбор признаков (RF, RFE)',
            'Метод главных компонент (PCA)',
            'Кластерный анализ',
            'Машинное обучение',
        ]

        cb_widgets = []
        display(widgets.HTML('<b>Выберите разделы для отчёта:</b>'))
        for k, lab in zip(sec_keys, sec_labels):
            cb = widgets.Checkbox(value=True, description=lab, indent=False, layout=widgets.Layout(width='320px'))
            cb_widgets.append(cb)
        display(widgets.VBox(cb_widgets))

        btn_gen = widgets.Button(
            description='📄 Сгенерировать HTML',
            button_style='success',
            layout=widgets.Layout(width='180px')
        )
        btn_reset = widgets.Button(
            description='🔄 Выбрать другие параметры',
            button_style='warning',
            layout=widgets.Layout(width='220px')
        )
        btn_open = widgets.Button(
            description='📂 Скачать HTML',
            button_style='info',
            layout=widgets.Layout(width='160px')
        )
        out_gen = widgets.Output()

        def on_gen(b):
            with out_gen:
                clear_output(wait=True)
                sel = {k: w.value for k, w in zip(sec_keys, cb_widgets)}
                analyzer.generate_html_report(df_clean, sections=sel)

        def on_reset(b):
            with out_gen:
                clear_output(wait=True)
            if 'analyzer' in globals():
                del globals()['analyzer']
            print('🔄 Параметры сброшены. Запустите ячейку снова для выбора новых параметров.')

        def on_open(b):
            with out_gen:
                clear_output(wait=True)
                import glob
                html_files = [f for f in glob.glob('*_report.html') if os.path.isfile(f)]
                if not html_files:
                    print('HTML-отчёт не найден. Сначала сгенерируйте его.')
                    return
                latest = max(html_files, key=os.path.getmtime)
                print(f'Скачиваю отчёт: {latest}')
                try:
                    from google.colab import files
                    files.download(latest)
                except ImportError:
                    from IPython.display import FileLink, display as _dsp
                    _dsp(FileLink(latest))

        btn_gen.on_click(on_gen)
        btn_reset.on_click(on_reset)
        btn_open.on_click(on_open)

        hbox = widgets.HBox([btn_gen, btn_reset, btn_open])
        display(widgets.VBox([hbox, out_gen]))

except Exception as e:
    import traceback
    print(f'❌ Ошибка: {e}')
    traceback.print_exc()



## 📥 Результаты

- **HTML-отчёт** — нажмите **«📂 Скачать HTML»** в ячейке анализа (Colab скачает файл в браузер).
- **Кластеры** — сохраняются в файл `*_with_clusters.xlsx`.
- Все промежуточные файлы лежат в **/content** (панель «Файлы» слева).

> 💡 **Совет:** если нужно просто открыть отчёт — после генерации нажмите «📂 Скачать HTML» и откройте скачанный файл в браузере.

